# ROGII - Baseline Models

This Kaggle-only notebook builds a small baseline model suite for the ROGII Wellbore Geology Prediction competition.

The goal is practical: compare simple, inference-safe per-well methods under masked-tail validation, choose the best baseline, and write `/kaggle/working/submission.csv`.

Baselines included:

- carry-forward `TVT_input`;
- per-well linear trend extrapolation from the known prefix;
- damped linear trend extrapolation;
- validation-selected blend of carry-forward and trend.

These methods use only test-time columns and avoid train-only geology-top leakage.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)

KAGGLE_INPUT_ROOT = Path('/kaggle/input')
COMPETITION_SLUG = 'rogii-wellbore-geology-prediction'
DATA_ROOT_CANDIDATES = [
    KAGGLE_INPUT_ROOT / 'competitions' / COMPETITION_SLUG,
    KAGGLE_INPUT_ROOT / COMPETITION_SLUG,
]
WORK_DIR = Path('/kaggle/working')
SUBMISSION_PATH = WORK_DIR / 'submission.csv'


def resolve_data_root(candidates):
    for candidate in candidates:
        if (candidate / 'sample_submission.csv').exists():
            return candidate
    for sample_file in KAGGLE_INPUT_ROOT.rglob('sample_submission.csv') if KAGGLE_INPUT_ROOT.exists() else []:
        if COMPETITION_SLUG in sample_file.as_posix():
            return sample_file.parent
    return candidates[0]


DATA_ROOT = resolve_data_root(DATA_ROOT_CANDIDATES)
print('DATA_ROOT:', DATA_ROOT)
print('sample_submission exists:', (DATA_ROOT / 'sample_submission.csv').exists())

## 1. Load Competition Files

The competition is organized by well. We discover horizontal well CSVs from `train/` and `test/`, then parse `sample_submission.csv` into `well` and `row_idx` so predictions can be mapped back into the required row order.

In [ ]:
def find_files(root: Path, pattern: str):
    return sorted(root.rglob(pattern)) if root.exists() else []


def well_name_from_horizontal_path(path: Path) -> str:
    return path.name.split('__horizontal_well.csv')[0]


def parse_submission_id(value):
    well, row = str(value).rsplit('_', 1)
    return well, int(row)


def get_column(df: pd.DataFrame, name: str):
    lookup = {col.lower(): col for col in df.columns}
    return lookup.get(name.lower())


train_horizontal_files = find_files(DATA_ROOT / 'train', '*__horizontal_well.csv')
test_horizontal_files = find_files(DATA_ROOT / 'test', '*__horizontal_well.csv')
sample_submission = pd.read_csv(DATA_ROOT / 'sample_submission.csv')

id_col = sample_submission.columns[0]
target_col = 'tvt' if 'tvt' in sample_submission.columns else sample_submission.columns[-1]
parsed_ids = sample_submission[id_col].map(parse_submission_id)
sample_submission['well'] = [item[0] for item in parsed_ids]
sample_submission['row_idx'] = [item[1] for item in parsed_ids]

print('train horizontal wells:', len(train_horizontal_files))
print('test horizontal wells:', len(test_horizontal_files))
print('submission rows:', len(sample_submission))
display(sample_submission.head())

## 2. Baseline Model Functions

All baselines operate per well. This keeps the first modeling pass robust to hidden test-size changes and avoids relying on train-only columns.

The key idea is to treat `TVT_input` as a known prefix. The hidden suffix is predicted by either holding the last known value constant or extending the recent per-well trend.

In [ ]:
def numeric_series(df: pd.DataFrame, column: str, default=np.nan):
    if column is None:
        return pd.Series(default, index=df.index, dtype='float64')
    return pd.to_numeric(df[column], errors='coerce').astype('float64')


def coordinate_axis(df: pd.DataFrame):
    md_col = get_column(df, 'MD')
    if md_col is not None:
        x = numeric_series(df, md_col)
        if x.notna().sum() >= 2:
            return x.reset_index(drop=True)
    return pd.Series(np.arange(len(df), dtype='float64'))


def tvt_input_series(df: pd.DataFrame):
    tvt_input_col = get_column(df, 'TVT_input')
    tvt_col = get_column(df, 'TVT')
    if tvt_input_col is not None:
        return numeric_series(df, tvt_input_col).reset_index(drop=True)
    if tvt_col is not None:
        return numeric_series(df, tvt_col).reset_index(drop=True)
    return pd.Series(np.nan, index=range(len(df)), dtype='float64')


def carry_forward_prediction(df: pd.DataFrame):
    y = tvt_input_series(df)
    pred = y.ffill().bfill()
    if pred.isna().all():
        pred = pd.Series(0.0, index=range(len(df)), dtype='float64')
    return pred.astype('float64')


def linear_trend_prediction(df: pd.DataFrame, tail_points=500):
    x = coordinate_axis(df).astype('float64')
    y = tvt_input_series(df).astype('float64')
    known = y.notna() & x.notna()
    carry = carry_forward_prediction(df)
    if known.sum() < 2:
        return carry

    x_known = x[known].to_numpy()
    y_known = y[known].to_numpy()
    n_tail = min(tail_points, len(x_known))
    x_tail = x_known[-n_tail:]
    y_tail = y_known[-n_tail:]
    x_anchor = x_tail[-1]
    y_anchor = y_tail[-1]

    if np.nanstd(x_tail) == 0:
        return carry

    slope = np.polyfit(x_tail - x_anchor, y_tail - y_anchor, 1)[0]
    pred = y_anchor + slope * (x.to_numpy() - x_anchor)
    pred = pd.Series(pred, index=range(len(df)), dtype='float64')
    pred[y.notna()] = y[y.notna()]
    return pred.ffill().bfill().astype('float64')


def damped_trend_prediction(df: pd.DataFrame, damp=0.35, tail_points=500):
    carry = carry_forward_prediction(df)
    trend = linear_trend_prediction(df, tail_points=tail_points)
    y = tvt_input_series(df)
    pred = carry + damp * (trend - carry)
    pred[y.notna()] = y[y.notna()]
    return pred.astype('float64')


def blended_prediction(df: pd.DataFrame, weight=0.50, tail_points=500):
    carry = carry_forward_prediction(df)
    trend = linear_trend_prediction(df, tail_points=tail_points)
    y = tvt_input_series(df)
    pred = (1 - weight) * carry + weight * trend
    pred[y.notna()] = y[y.notna()]
    return pred.astype('float64')


def predict_by_model(df: pd.DataFrame, model_name: str):
    if model_name == 'carry_forward':
        return carry_forward_prediction(df)
    if model_name == 'linear_trend':
        return linear_trend_prediction(df)
    if model_name == 'damped_trend_035':
        return damped_trend_prediction(df, damp=0.35)
    if model_name.startswith('blend_'):
        weight = float(model_name.split('_')[1])
        return blended_prediction(df, weight=weight)
    raise ValueError(f'Unknown model: {model_name}')

## 3. Masked-Tail Validation

Validation should mimic the competition shape: keep a prefix of true `TVT`, hide the suffix, and score predictions on the hidden rows.

This validation is still a simplification. It tests extrapolation behavior, but it does not fully reproduce hidden-test well diversity or all geological discontinuities.

In [ ]:
def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype='float64')
    y_pred = np.asarray(y_pred, dtype='float64')
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    return float(np.sqrt(np.mean((y_true[mask] - y_pred[mask]) ** 2))) if mask.any() else np.nan


def make_masked_validation_frame(df: pd.DataFrame, tail_fraction=0.25):
    tvt_col = get_column(df, 'TVT')
    if tvt_col is None:
        return None, None
    y_true = numeric_series(df, tvt_col).reset_index(drop=True)
    eval_start = int(len(df) * (1 - tail_fraction))
    masked = df.copy().reset_index(drop=True)
    masked['TVT_input'] = y_true.copy()
    masked.loc[eval_start:, 'TVT_input'] = np.nan
    return masked, y_true


def evaluate_baselines(files, max_wells=120, tail_fractions=(0.20, 0.25, 0.30)):
    model_names = ['carry_forward', 'linear_trend', 'damped_trend_035', 'blend_0.25', 'blend_0.50', 'blend_0.75']
    records = []

    for path in files[:max_wells]:
        well = well_name_from_horizontal_path(path)
        df = pd.read_csv(path)
        for tail_fraction in tail_fractions:
            masked, y_true = make_masked_validation_frame(df, tail_fraction=tail_fraction)
            if masked is None:
                continue
            eval_start = int(len(masked) * (1 - tail_fraction))
            for model_name in model_names:
                pred = predict_by_model(masked, model_name)
                records.append({
                    'well': well,
                    'rows': len(masked),
                    'tail_fraction': tail_fraction,
                    'model': model_name,
                    'rmse': rmse(y_true.iloc[eval_start:], pred.iloc[eval_start:]),
                })

    return pd.DataFrame(records)


validation = evaluate_baselines(train_horizontal_files)
summary = validation.groupby('model')['rmse'].agg(['mean', 'median', 'std', 'count']).sort_values('mean') if not validation.empty else pd.DataFrame()
display(summary)
display(validation.head())

## 4. Select Baseline And Build Submission

The selected model is the lowest mean RMSE from masked-tail validation. If validation cannot run, the notebook falls back to the conservative carry-forward baseline.

The final submission is written in exactly the same row order as `sample_submission.csv`.

In [ ]:
selected_model = 'carry_forward'
if not summary.empty:
    selected_model = summary.index[0]

print('selected_model:', selected_model)

horizontal_lookup = {well_name_from_horizontal_path(path): path for path in test_horizontal_files}
submission = sample_submission[[id_col]].copy()
predicted_tvt = []
missing_wells = set()

fallback = 0.0
well_prediction_cache = {}
for well, path in horizontal_lookup.items():
    df = pd.read_csv(path)
    pred = predict_by_model(df, selected_model).reset_index(drop=True)
    well_prediction_cache[well] = pred
    if pred.notna().any():
        fallback = float(pred.dropna().median())

for sub_id in sample_submission[id_col]:
    well, row_idx = parse_submission_id(sub_id)
    pred = well_prediction_cache.get(well)
    if pred is None or len(pred) == 0:
        missing_wells.add(well)
        predicted_tvt.append(fallback)
    elif 0 <= row_idx < len(pred):
        predicted_tvt.append(float(pred.iloc[row_idx]))
    else:
        predicted_tvt.append(float(pred.iloc[-1]))

submission[target_col] = predicted_tvt
submission.to_csv(SUBMISSION_PATH, index=False)

print('wrote:', SUBMISSION_PATH)
print('rows:', len(submission))
print('missing wells:', len(missing_wells))
display(submission.head())
display(submission[target_col].describe())

## 5. Interpretation

Use the validation summary to decide whether trend extrapolation is actually improving over carry-forward. If trend models perform worse, it usually means the known prefix slope does not persist through the hidden interval.

A strong next step is to add `GR` and typewell-alignment features under the same masked-tail validation protocol, then compare every new model against this notebook's selected baseline.